# Lab 11: SQL for Economists — The Same Operations, A Different Grammar
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 80 min Core + 30 min AI + 15 min Extension + take-home Challenge

---

**Learning Objectives:**
- Translate the pandas operations you know into SQL, clause by clause, and see that they are the same relational algebra
- Diagnose the cardinality trap: a join that silently multiplies rows and corrupts every statistic computed after it
- Use window functions to aggregate without collapsing rows
- Query a remote file without downloading it, reading only the columns the query needs
- Check SQL that AI translated from your pandas, by comparing row counts

**Dataset:** Synthetic customers and orders, plus NYC Taxi & Limousine Commission trip records (public Parquet, about 3 million rows, queried where it sits)

**Prerequisites:** the Part 0 sections of Labs 1-5, and the labs before this one.

**How this lab works:** Parts 1-3 are GUIDED: run each cell as it is, then answer the questions. Part 4 has one blank, in Step 7. The Extension is optional, and the Challenge is a take-home. The AI-Assisted Expansion at the end is the one place you use AI to write code, and you check what it writes against your own numbers.

**AI in the manual parts:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion while you work on those parts. AI-assisted coding is allowed only where this lab says so.

---

> ### What you write, and what you run
>
> **You write:**
> - **Part 4, Step 7:** one blank (`____`) inside a SQL query: the `WHERE` condition the step names. Predict the row count before you run it.
> - The answers to the interpretation questions after each part, in text cells. Part 1's Question 1 also asks for a few lines of pandas, in a new code cell.
> - **The AI Expansion:** version 2 of the given prompt, revised once, after you find what version 1 left the AI to guess. Then four short answers: your revision, the row-count check against your own pandas results from Steps 3, 4 and 7, the key lines in your own words, and when AI is worth using for this.
>
> **You run and read:** everything else. Setup, Parts 1-3, the rest of Part 4, and the check cell in the AI Expansion are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The AI's queries are pasted, not typed, and the queries themselves are not graded; your answers about them are.
>
> **The Extension** (optional) has two `None` values to replace with a short piece of SQL each; untouched, it prints "Extension not attempted." **The Challenge** (take-home) has no starter code: you run a given query, and write your answers and one rewritten query in new cells. The **README** is text only: the prompt at the end is for documentation, not code.

## Setup

In [ ]:
# GUIDED — run as-is
# Step 1: install DuckDB, then import libraries
!pip install -q duckdb

import numpy as np
import pandas as pd
import duckdb
from scipy import stats

# Seed 7: the same tables on every run, so your numbers match the expected output
rng = np.random.default_rng(7)

print(f"DuckDB {duckdb.__version__} ready.")

## Part 1: You Already Know This (GUIDED — 25 min)

SQL is not a different way of thinking about data. It is the same relational algebra you
have been doing in pandas since Chapter 1, written in the syntax that every database,
warehouse and co-op interview expects. A `groupby` is a `GROUP BY`. A `merge` is a `JOIN`.

In [ ]:
# GUIDED — run as-is
# Step 2: build two tables, customers and their orders
n_cust = 600
customers = pd.DataFrame({
    "customer_id": np.arange(n_cust),                  # 0, 1, ..., 599
    "region": rng.choice(["East", "West"], n_cust),    # each customer East or West at random
    "revenue": rng.normal(500, 120, n_cust).round(2),  # the same distribution in both regions
})

# Orders are one-to-many: each customer gets a Poisson number of orders, 6 on average
n_orders = rng.poisson(6, n_cust)
# np.repeat writes each customer_id once per order (.values gives the column as a NumPy array)
orders = pd.DataFrame({
    "customer_id": np.repeat(customers["customer_id"].values, n_orders),
})
# gamma draws are positive and right-skewed, like order values
orders["order_value"] = rng.gamma(2, 40, len(orders)).round(2)

# :, puts a comma between thousands, so 3612 prints as 3,612
print(f"customers: {len(customers):,} rows (one per customer)")
print(f"orders:    {len(orders):,} rows (several per customer)")
customers.head()

**Expected output:** 600 customers and 3,612 orders, about six per customer, then the first five
rows of `customers`.

### First: where is the database?

There is no database server in this lab. That is worth knowing before you read a line
of SQL, because the queries below look as if they read from a database called `customers`.

`duckdb.sql("SELECT ... FROM customers")` reads the name `customers` out of your
**Python namespace** — the same variable you built one cell ago with
`pd.DataFrame(...)` — and runs the query directly against that DataFrame, in
memory. No server, no connection string, no `CREATE TABLE`, no file on disk.
DuckDB is a SQL engine you imported, not a database you connected to.

Two consequences to hold on to:

- **The table name in the query *is* the Python variable name.** Rename the
  variable and the SQL breaks with `Table with name customers does not exist`.
- **`.df()` on the end converts the answer back into a pandas DataFrame.**
  Without it you get a DuckDB relation, which prints nicely but is not a
  DataFrame and will not accept `.groupby()` or `.iloc`.

Almost everything you write below is real, portable SQL (Step 8's `read_parquet` is
DuckDB's own). The same text would run
against Postgres, Snowflake or BigQuery, with the table names pointing at real
tables instead of at your notebook's variables.

### The translation table

You have been doing relational algebra since Lab 1. Here is the dictionary.

| pandas | SQL | what it does |
|---|---|---|
| `df[["a", "b"]]` | `SELECT a, b FROM df` | pick columns |
| `df[df["a"] > 5]` | `... WHERE a > 5` | pick rows, before grouping |
| `df.sort_values("a", ascending=False)` | `... ORDER BY a DESC` | sort |
| `df.head(10)` | `... LIMIT 10` | keep the first n rows |
| `df.groupby("g")["x"].mean()` | `SELECT g, AVG(x) ... GROUP BY g` | split-apply-combine |
| filtering on the aggregate afterwards | `... HAVING AVG(x) > 5` | pick rows **after** grouping |
| `df.merge(other, on="k", how="left")` | `... LEFT JOIN other ON ...` | combine two tables |
| `df["x"].nunique()` | `COUNT(DISTINCT x)` | count distinct values |
| `df.rename(columns={"x": "y"})` | `SELECT x AS y` | rename a column |
| `len(df)` | `SELECT COUNT(*) FROM df` | count rows |

There is one structural thing pandas does not prepare you for. SQL clauses must
be **written** in a fixed order, and it is not the order they **run** in:

```
SELECT   ...     -- written 1st, runs 5th: which columns come out
FROM     ...     -- written 2nd, runs 1st: which table
WHERE    ...     -- written 3rd, runs 2nd: filter rows
GROUP BY ...     -- written 4th, runs 3rd: form the groups
HAVING   ...     -- written 5th, runs 4th: filter the groups
ORDER BY ...     -- written 6th, runs 6th: sort
LIMIT    ...     -- written 7th, runs 7th: cut
```

Two things fall straight out of that. `WHERE` filters individual rows and
`HAVING` filters groups, because `WHERE` has already finished by the time the
groups exist. And in most databases you cannot use a name you invented in
`SELECT` inside a `WHERE`, for the same reason: the alias does not exist yet when
`WHERE` runs. (DuckDB is lenient and allows it, so code that works here can fail
on another database.)

In [ ]:
# GUIDED — run as-is
# Step 2b: four operations, each written both ways

def compare(label, pandas_result, sql_text):
    """Print a pandas result above the same query's result in SQL."""
    print(f"--- {label} ---")
    print("pandas:")
    # index=False prints the table without pandas' row numbers, which SQL does not have
    print(pandas_result.to_string(index=False))
    print("SQL:")
    print(duckdb.sql(sql_text).df().to_string(index=False))
    print()

# 1. SELECT: pick columns.
compare(
    "pick two columns",
    customers[["customer_id", "revenue"]].head(3),
    "SELECT customer_id, revenue FROM customers ORDER BY customer_id LIMIT 3",
)

# 2. WHERE: pick rows. A string in triple quotes can run over several lines.
compare(
    "rows with revenue over 700",
    customers[customers["revenue"] > 700][["customer_id", "revenue"]].head(3),
    """SELECT customer_id, revenue FROM customers
       WHERE revenue > 700 ORDER BY customer_id LIMIT 3""",
)

# 3. ORDER BY: sort. DESC is descending; ASC is the default.
compare(
    "three highest revenues",
    customers.sort_values("revenue", ascending=False)[["customer_id", "revenue"]].head(3),
    "SELECT customer_id, revenue FROM customers ORDER BY revenue DESC LIMIT 3",
)

# 4. HAVING: filter AFTER grouping. pandas aggregates first, .reset_index() turns region back
#    into a column, and .query() keeps the rows where the condition is true
compare(
    "regions averaging over 490.50",
    (customers.groupby("region")["revenue"].mean().round(2)
     .reset_index().query("revenue > 490.50")),
    """SELECT region, ROUND(AVG(revenue), 2) AS revenue
       FROM customers
       GROUP BY region
       HAVING AVG(revenue) > 490.50
       ORDER BY region""",
)

**Expected output:** four pairs of tables, and in each pair the pandas rows and the SQL rows are
the same. Only East's average, 490.91, is above 490.50, so the `HAVING` query keeps one region.

In [ ]:
# GUIDED — run as-is
# Step 3: the same question, asked twice
# .agg(["count", "mean"]) computes both statistics for each region
pandas_way = (customers.groupby("region")["revenue"]
              .agg(["count", "mean"]).round(2).reset_index())

sql_way = duckdb.sql("""
    SELECT region, COUNT(*) AS count, ROUND(AVG(revenue), 2) AS mean
    FROM customers
    GROUP BY region
    ORDER BY region
""").df()

print("pandas:")
print(pandas_way.to_string(index=False))
print("\nSQL:")
print(sql_way.to_string(index=False))

**Expected output:** the same table twice. East has 282 customers with a mean revenue of 490.91,
and West has 318 with 490.22. The pandas `groupby` and the SQL
`GROUP BY` give the same table.

### Interpretation Questions

1. Write this query in pandas, one step per clause: `SELECT region, AVG(revenue) FROM customers WHERE revenue > 400 GROUP BY region HAVING COUNT(*) > 230 ORDER BY region`. Which pandas step does the job of `WHERE`, and which the job of `HAVING`? Run both versions to check that they agree.
2. SQL has no row order unless you ask for it, while a DataFrame always has an index. Give one bug that difference could cause.
3. SQL makes you name `region` in both `SELECT` and `GROUP BY`. Why?

*Your answers here:*

## Part 2: The Cardinality Trap (GUIDED — 20 min)

This is the part of the chapter with real consequences. A join is not a lookup. When the
right-hand table has several rows per key, the left-hand rows are **repeated** — and every
statistic you compute afterwards is silently weighted by order count.

### Joins, and the two-letter names in them

`FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id` is
`customers.merge(orders, on="customer_id", how="left")`. Four pieces:

- **`customers c`**: `c` is an **alias**, a nickname for the table. You declare
  one just by writing it after the table name, and from then on you can say
  `c.revenue` instead of `customers.revenue`. It is pure convenience, and it
  becomes necessary the moment two tables share a column name, which is exactly
  what a join key is.
- **`LEFT JOIN orders o`**: the second table, aliased `o`.
- **`ON c.customer_id = o.customer_id`**: the **join key**: the column that has
  to match for two rows to be glued together. Name the wrong column here and you
  get a wrong answer, not an error.
- **`LEFT`**: keep every row of the left table even when the right table has no
  match, filling the missing side with NULL. That is `how="left"`. `INNER JOIN`
  (the bare word `JOIN`) keeps only rows that matched; `FULL JOIN` keeps
  everything from both sides.

Run the next cell first. It has one row per region on the right-hand side, so nothing
multiplies. Then Step 4 shows you what happens when the right-hand side has
several rows per key, which is the part with consequences.

In [ ]:
# GUIDED — run as-is
# Step 3b: one join, written both ways
region_targets = pd.DataFrame({"region": ["East", "West"], "target": [520, 480]})

pandas_join = (customers[customers["customer_id"] < 4]
               .merge(region_targets, on="region", how="left")
               [["customer_id", "region", "revenue", "target"]])

sql_join = duckdb.sql("""
    SELECT c.customer_id, c.region, c.revenue, t.target
    FROM customers c
    LEFT JOIN region_targets t ON c.region = t.region
    WHERE c.customer_id < 4
    ORDER BY c.customer_id
""").df()

print("pandas .merge():")
print(pandas_join.to_string(index=False))
print("\nSQL LEFT JOIN:")
print(sql_join.to_string(index=False))

**Expected output:** the same four rows twice. All four customers are in the West, so each gets
the West target of 480. Four rows in, four rows out: `region_targets` has exactly one row per
region, so nothing was duplicated. Step 4 joins to a table with several rows per
key.

In [ ]:
# GUIDED — run as-is
# Step 4: join customers to orders, and count the rows
fanned = duckdb.sql("""
    SELECT c.customer_id, c.region, c.revenue
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
""").df()

# The same join in pandas
fanned_pandas = customers.merge(orders, on="customer_id", how="left")

print(f"Before the join: {len(customers):,} rows")
print(f"After the join:  {len(fanned):,} rows")
print(f"pandas merge:    {len(fanned_pandas):,} rows")

**Expected output:** 600 rows before the join and 3,612 after it, in SQL and in pandas alike. Every
customer now appears once per order, with `revenue` repeated on each of those rows. The join
runs without an error or a warning.

In [ ]:
# GUIDED — run as-is
# Step 5: the same Welch t-test on both tables
def welch(frame, label):
    """Welch t-test of revenue, East against West, printed on one line."""
    # .loc[rows, column]: the revenue column, for the East rows
    east = frame.loc[frame["region"] == "East", "revenue"]
    west = frame.loc[frame["region"] == "West", "revenue"]
    # equal_var=False makes it Welch's test, which does not assume the two variances are equal
    t, p = stats.ttest_ind(east, west, equal_var=False)
    diff = east.mean() - west.mean()
    se = diff / t          # t is the difference divided by its standard error
    # <34 pads the label to 34 characters, >6, right-aligns n in 6 with a comma, .2f keeps 2 decimals
    print(f"{label:<34} n = {len(frame):>6,}   diff = {diff:>5.2f}   SE = {se:>5.2f}"
          f"   t = {t:>6.2f}   p = {p:.4f}")

welch(customers, "One row per customer (correct)")
welch(fanned, "After the join (fanned out)")

**Expected output:** with one row per customer, East's mean revenue is 0.69 above West's, with a
standard error of 9.35: t = 0.07, p = 0.9413. After the join, n is 3,612, and t = 0.71, p = 0.4749.

The revenue figures are the same in both tables. The join changed only how many times each one is
counted, and that did two things:
- **The standard error shrank** from 9.35 to 3.75, about 2.5 times. Each customer appears about
  six times (3,612 / 600 = 6.02), and √6.02 ≈ 2.45: the test thinks it has six times the data.
- **The difference itself moved**, from 0.69 to 2.68. Question 2 asks why.

Together they raised t about tenfold. On this draw neither p-value is below 0.05: both regions'
revenue was drawn from the same distribution, so there is no difference to find. Where a real
difference exists, the same shrinking of the standard error is what turns a marginal p = 0.06
into p < 0.001, and nothing in the output tells you the sample size is a fiction.

### Interpretation Questions

1. Explain in one sentence why repeating a customer's revenue six times shrinks the p-value, even though no new customer was surveyed.
2. Which customers get the most weight after the fan-out? Say what that does to the estimate if heavy orderers differ systematically.
3. You are handed a query with three joins and a `COUNT(*)` of 40,000 on a table you know has 600 customers. What do you check first?

*Your answers here:*

## Part 3: Aggregating Without Collapsing (GUIDED — 15 min)

`GROUP BY` answers "what is the average per region" but destroys the rows. A **window
function** computes the same aggregate and keeps every row — which is what you want when
the comparison itself is the variable of interest.

### What `OVER (PARTITION BY ...)` is

`GROUP BY` gives you one row per group and throws the individual rows away. A
**window function** computes the same aggregate but attaches it to every row,
so nothing is lost.

`AVG(revenue) OVER (PARTITION BY region)` reads: *average revenue, computed
separately within each region, reported on every row of that region.*

The pieces:

- **`OVER (...)`** is what turns an ordinary aggregate into a window function.
  `SUM`, `COUNT`, `MIN`, `MAX` all take it.
- **`PARTITION BY region`** is the grouping. Omit it and the window is the
  entire table.
- **`ORDER BY revenue DESC`** *inside* the `OVER (...)` orders rows within the
  window. That is what ranking needs, and it is unrelated to the query's own
  `ORDER BY` at the bottom, which only controls how the output is printed.
- **`RANK()`** is a window-only function: it numbers the rows within their
  window, 1 for the top.

In [ ]:
# GUIDED — run as-is
# Step 6: each customer against their own region's average
windowed = duckdb.sql("""
    SELECT
        customer_id,
        region,
        revenue,
        ROUND(AVG(revenue) OVER (PARTITION BY region), 2) AS region_avg,
        ROUND(revenue - AVG(revenue) OVER (PARTITION BY region), 2) AS vs_region,
        RANK() OVER (PARTITION BY region ORDER BY revenue DESC) AS rank_in_region
    FROM customers
    ORDER BY region, rank_in_region
""").df()

print(f"Rows in: {len(customers):,}   rows out: {len(windowed):,}")
print(windowed.head(8).to_string(index=False))      # the first 8 rows: East's top 8

**Expected output:** 600 rows in and 600 out. East's top customer is customer 120, with revenue of
804.43: 313.52 above East's average of 490.91, so ranked 1. Every row carries its region's average.

### Interpretation Questions

1. Write the pandas equivalent of `AVG(revenue) OVER (PARTITION BY region)`. Which is easier to read?
2. `RANK()` and `ROW_NUMBER()` differ on ties. Say how, and give a case where the choice matters.
3. Why can a window function not go in a `WHERE` clause? What runs instead?

*Your answers here:*

## Part 4: NULL Is Not a Value, and Data You Never Downloaded (YOUR TASK — 20 min)

Two things that surprise people arriving from pandas: SQL's **three-valued logic**, which means
a condition can be true, false or unknown, and the fact
that you can query a file sitting on someone else's server without fetching all of it.
Step 7 has one blank; the rest is guided.

In [ ]:
# GUIDED — run as-is
# Step 7: NULL propagates through comparisons
# None marks a missing income; 40_000 is 40000, and the underscore only makes it easier to read
survey = pd.DataFrame({
    "respondent": [1, 2, 3, 4, 5],
    "income": [40_000, None, 65_000, None, 90_000],
})

# COUNT(*) counts rows; COUNT(income) counts the incomes that are not NULL
print(duckdb.sql("""
    SELECT
        COUNT(*)        AS total_rows,
        COUNT(income)   AS non_null_incomes,
        AVG(income)     AS avg_income
    FROM survey
""").df().to_string(index=False))

**Expected output:** 5 rows, 3 incomes that are not NULL, and an average of 65,000.0. `AVG` skipped
the two missing incomes and divided by 3, not 5.

SQL tests for a missing value with its own words: `income IS NULL`. A comparison with a NULL,
such as `income = NULL` or `income != 50000` on a row with no income, is unknown, not true
or false, and `WHERE` keeps only the rows where the condition is true.

In [ ]:
# YOUR TASK — fill in the blank (____)
# Step 7, continued: how many rows does WHERE income != 50000 keep?
# Predict the count before you run it. Then put that condition in the blank.
answer = duckdb.sql("""
    SELECT COUNT(*) AS n FROM survey WHERE ____
""").df()

# answer["n"][0] is the first, and only, value in the n column
print(f"Rows where income != 50000: {answer['n'][0]}")

**Expected output:** one line with the count. Compare it with your prediction before you read on.
*Binder Error: Referenced column "____" not found* means the blank is still empty.

In [ ]:
# GUIDED — run as-is
# Step 7, continued: the same filter in pandas
kept_pandas = survey[survey["income"] != 50000]
print(f"Rows pandas keeps: {len(kept_pandas)}")

**Expected output:** pandas keeps all 5 rows. In pandas, a missing value compared with `!=` counts
as "not equal", so the two respondents with no income stay in. Your SQL query kept 3. Same
condition, same data, two answers: Question 1 asks why.

In [ ]:
# GUIDED — run as-is (it needs an internet connection, and takes a few seconds)
# Step 8: query 3 million rows you never downloaded
import time

# A connection of your own, so you can switch on httpfs, the extension that reads files over the web
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# NYC Taxi & Limousine Commission trip records, January 2024. Public Parquet.
url = ("https://d37ci6vzurychx.cloudfront.net/trip-data/"
       "yellow_tripdata_2024-01.parquet")

# time.time() reads the clock in seconds; the difference afterwards is how long the query took
t0 = time.time()
# An f-string can hold a query too: {url} puts the address into the SQL
result = con.execute(f"""
    SELECT
        passenger_count,
        COUNT(*)                       AS trips,
        ROUND(AVG(trip_distance), 2)   AS avg_miles,
        ROUND(AVG(total_amount), 2)    AS avg_fare
    FROM read_parquet('{url}')
    WHERE passenger_count BETWEEN 1 AND 6
      AND trip_distance > 0
    GROUP BY passenger_count
    ORDER BY passenger_count
""").df()
elapsed = time.time() - t0

# formatters prints the trips column with commas, as {:,} does in an f-string
print(result.to_string(index=False, formatters={"trips": "{:,}".format}))
print(f"\nElapsed: {elapsed:.1f}s")

**Expected output:** six rows, one per passenger count from 1 to 6. One-passenger trips dominate:
2,159,728 of them, averaging 3.18 miles and a \$26.17 fare. The elapsed time is yours alone: it
depends on your connection and your machine. Time it yourself; a number you generated is the only
one you can defend.

The file holds about 3 million trips in 19 columns, about 50 MB on the server. DuckDB did not fetch
all of it: it read the three columns the query uses, applied the filters as it read them, and
brought back six rows.

### Interpretation Questions

1. Your `!= 50000` query returned 3 rows, and the same filter in pandas kept 5. None of the five incomes is 50,000. Explain what SQL did with the two missing incomes, and write a `WHERE` clause that returns 5.
2. Your timing came from one machine on one connection. Name two things that would change it, and say why the chapter refuses to quote a benchmark number.
3. The query returned six rows from three million. What did DuckDB avoid doing, and why does that make "bigger than RAM" the wrong way to think about the limit?

*Your answers here:*

---

## Extension (Optional — 15 min): Fix the Fan-Out

Part 2 broke a t-test with a join. Aggregate *before* joining and the problem disappears.
This is the single most useful defensive habit in analytical SQL.

The query below has two pieces you have not met:
- `WITH order_summary AS (...)` names an intermediate result, a **common table expression**
  (CTE), and the main query then uses `order_summary` like a table. It is the same move as
  `summary = orders.groupby(...)` in pandas. Here it collapses `orders` to one row per customer
  before the join, so the join cannot fan out.
- `COALESCE(x, 0)` gives 0 where `x` is NULL. A customer with no orders has no row in
  `order_summary`, so the `LEFT JOIN` would give that customer NULL counts (none in this draw; real data has
  them).

Fill in the two aggregates: replace each `None` with a short piece of SQL, in quotes. The
f-string puts them into the `WITH` block, as Step 8 put `{url}` into its query. Part 1's
translation table and Part 3's notes on window functions name the aggregate functions SQL
has. Until both are filled in, the cell prints "Extension not attempted." and the notebook
carries on.

In [ ]:
# YOUR TASK (optional) — replace the two None values
# Step 9: aggregate first, then join
n_orders_sql = None        # SQL, in quotes, for each customer's number of orders
total_value_sql = None     # SQL, in quotes, for the total value of those orders

if n_orders_sql is None or total_value_sql is None:
    print("Extension not attempted.")
else:
    safe = duckdb.sql(f"""
        WITH order_summary AS (
            SELECT customer_id,
                   {n_orders_sql} AS n_orders,
                   {total_value_sql} AS total_value
            FROM orders
            GROUP BY customer_id
        )
        SELECT c.customer_id, c.region, c.revenue,
               COALESCE(s.n_orders, 0)    AS n_orders,
               COALESCE(s.total_value, 0) AS total_value
        FROM customers c
        LEFT JOIN order_summary s ON c.customer_id = s.customer_id
    """).df()

    print(f"Rows: {len(safe):,}  (should be {len(customers):,})")
    welch(safe, "Aggregate-then-join")

**Expected output:** 600 rows, one per customer, and a t-test line that matches Step 5's
one-row-per-customer line exactly: diff = 0.69, SE = 9.35, t = 0.07, p = 0.9413. Each customer is
counted once. "Extension not attempted." means a `None` is still there; a *Binder Error* means
one of your two pieces of SQL names something the `orders` table does not have.

---

## Challenge (Take-Home): Read a Query You Did Not Write

Most SQL you meet in a job is inherited. The skill is not writing it — it is working out
what an existing query does, and whether it is right.

The cell below stores a query as text in `mystery`; running the cell does not run the query.
There is no other starter code. Work through the five steps, and write your answers, and your
code, in new cells below it.

1. Before running it, predict which of the four numbers are correct and which are corrupted by
   the join. Write your prediction down.
2. Run it with `duckdb.sql(mystery).df()`. Check your prediction against Step 3's count and mean
   revenue for each region.
3. Exactly ONE of `avg_revenue` and `total_orders` is trustworthy. Say which, and why the other
   is not.
4. Rewrite the query so every column is correct. State what you changed, and why
   `COUNT(DISTINCT ...)` was not enough on its own.
5. This query would pass code review at most companies. Say what a reviewer would have to know
   to catch it.

In [ ]:
# GUIDED — run as-is: it stores the Challenge's query as text, and does not run it
mystery = """
    SELECT c.region,
           COUNT(DISTINCT c.customer_id) AS customers,
           COUNT(*)                      AS rows_out,
           AVG(c.revenue)                AS avg_revenue,
           SUM(o.order_value)            AS total_orders
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.region
"""

---

## AI-Assisted Expansion: SQL translated by AI, checked by you

Three steps in this lab also ran in pandas: Step 3's `groupby`, Step 4's `merge` and Step 7's
filter. Translating pandas into SQL is a job AI does in seconds, and it is a fair test of AI,
because you already know how many rows each step returns. Ask an AI to translate those three
pandas steps into SQL, run its queries, and compare their row counts with your pandas results.

**What to do**
1. Copy **prompt version 1** below into Claude ([claude.northeastern.edu](https://claude.northeastern.edu)) or ChatGPT.
2. Read what comes back before you run anything. The prompt leaves at least one
   important thing unsaid, so the AI had to guess. Find one guess that is wrong,
   or one that you would rather decide yourself. Pasting the version 1 queries into
   the empty code cell and running the check is one way to find it.
3. Write **prompt version 2** in the cell below, fixing that one thing, and ask
   again. Paste the queries from version 2 into the empty code cell and run the
   check. If all three counts match, find where the AI decided what to do about
   missing incomes, and make version 2 state that choice yourself.
4. Answer the questions at the end. They are what is graded, not the queries.

**Reading the check.** It runs each of the AI's queries with DuckDB and prints its row count
beside the row count of your pandas result: `pandas_way` from Step 3, `fanned_pandas` from
Step 4 and `kept_pandas` from Step 7. It looks for the names `sql_step3`, `sql_step4` and
`sql_step7`, so keep those names. Paste only those three strings: if the AI also wrote example tables,
leave them out, because they would replace yours. A query that does not run gets DuckDB's
error message in place of a count.

A matching row count is a first check, not a proof: two queries can return the same number of
rows with different numbers in them. To look further, print one of the AI's results, for example
`duckdb.sql(sql_step3).df()`, and compare it with Step 3's table.

In [ ]:
# PROMPT, version 1 — copy it exactly as it is
#
# I am learning SQL with DuckDB in Python. I have three pandas DataFrames:
# customers (customer_id, region, revenue), orders (customer_id, order_value)
# and survey (respondent, income). Translate these three pandas steps into SQL
# that duckdb.sql() can run on the DataFrames by name:
#
#   Step 3: customers.groupby("region")["revenue"].agg(["count", "mean"]).round(2).reset_index()
#   Step 4: customers.merge(orders, on="customer_id", how="left")
#   Step 7: survey[survey["income"] != 50000]
#
# Give each query as a Python string called sql_step3, sql_step4 and sql_step7.
# Comment each line.

# PROMPT, version 2 — your revision. Keep everything from version 1 that was
# fine, and add or change the part the AI had to guess.
#
#

In [ ]:
# Paste the three queries from your version 2 here (sql_step3, sql_step4 and sql_step7), then run the check below.

In [ ]:
# CHECK — runs the AI's three queries and compares their row counts with your pandas results
# Each entry: the step, the name the AI's query must have, and your pandas result for that step
steps = [("Step 3, groupby", "sql_step3", pandas_way),
         ("Step 4, merge", "sql_step4", fanned_pandas),
         ("Step 7, filter", "sql_step7", kept_pandas)]

# The lab's tables have 600, 3,612 and 5 rows. Example tables pasted from the AI would replace them
tables_unchanged = len(customers) == 600 and len(orders) == 3612 and len(survey) == 5

# globals() holds every name defined so far in this notebook; globals()[name] looks one up by name.
# _ stands for the parts of each entry this line does not use
if not tables_unchanged:
    print("Your tables changed: paste only the three sql_step strings, then rerun from Step 2.")
elif not any(name in globals() for _, name, _ in steps):
    print("Paste the AI's three queries in the cell above first.")
else:
    print(f"{'':<16} {'pandas rows':>11}   AI query rows")
    for step, name, pandas_result in steps:
        start = f"{step:<16} {len(pandas_result):>11,}   "
        if name not in globals():
            print(start + f"no {name} pasted")
            continue
        query = globals()[name]
        # isinstance(query, str) is True when query is text
        if not isinstance(query, str):
            print(start + f"{name} should be the query text in quotes, not the result of running it")
            continue
        # try runs the query; if DuckDB raises an error, the lines under except run instead of stopping the cell
        try:
            print(start + f"{len(duckdb.sql(query).df()):,}")
        except duckdb.Error as err:
            print(start + "did not run: " + str(err).splitlines()[0])      # the first line of DuckDB's message

    print("\nIf a pair differs, find out why before you trust the query.")

### Document the interaction

This is what is graded for the expansion.

1. **Your revision.** What did you change between version 1 and version 2, and
   what did the AI get wrong or have to guess that made you change it? If every
   count matched the first time, name the choice the AI made on its own, and say
   how version 2 now states it.
2. **The check.** Which pairs of row counts matched, and which differed at first?
   For a pair that differed, what was the cause, and which count is the one the
   translation should give? How do you know?
3. **In your own words.** Pick the `WHERE` line of the AI's `sql_step7` and the
   JOIN line (`ON` or `USING`) of its `sql_step4`. Explain in 2–3 sentences what each does,
   and point to the pandas code it replaces.
4. **Would you use AI for this again?** One sentence: when is it worth asking AI
   to translate pandas into SQL, and when is it faster to write the query yourself?

*Your answers here:*

1.
2.
3.
4.

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Translated pandas operations (select, filter, sort, groupby, merge)
  into SQL with DuckDB, and checked that both give the same rows
* Showed the cardinality trap: a one-to-many LEFT JOIN turned
  [YOUR VALUE] customer rows into [YOUR VALUE] rows. The same Welch
  t-test on the same revenue figures went from t = [YOUR VALUE]
  (p = [YOUR VALUE]) to t = [YOUR VALUE] (p = [YOUR VALUE]), because
  the repeated rows shrank the standard error and moved the difference
* Used window functions (AVG OVER PARTITION BY, RANK) to compute
  group averages and ranks without collapsing rows
* Showed SQL's three-valued logic: WHERE income != 50000 kept
  [YOUR VALUE] of 5 survey rows, because [YOUR one-line reason],
  while the same filter in pandas kept [YOUR VALUE]
* Queried a public Parquet file of about 3 million NYC taxi trips
  over HTTP without downloading it, reading only the columns the
  query needed
* Asked an AI to translate three of my pandas steps into SQL,
  revised my prompt once, and compared the row counts of its
  queries with my pandas results

**Please write a README.md entry including:**
1. Project Title: SQL for Economists — Joins, Windows and the Cardinality Trap
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab11-sql`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab11-sql`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab11-sql`, branch `main`,
   commit message `Lab 11 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.